In [36]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.preprocessing import StandardScaler,MinMaxScaler,FunctionTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import NotFittedError
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,FeatureUnion
from sklearn.linear_model import LogisticRegression

In [37]:
TRAIN_PATH = Path.cwd() / "train"
TEST_PATH = Path.cwd() / "test"

In [38]:
# Load Independent set  
X_train = pd.read_csv(TRAIN_PATH / "X_train.csv")
X_test = pd.read_csv(TEST_PATH / "X_test.csv")

# Load Dependent set
y_train = pd.read_csv(TRAIN_PATH / "y_train.csv").to_numpy().ravel()
y_test  = pd.read_csv(TEST_PATH  / "y_test.csv").to_numpy().ravel()

# Will later be updated by a try-catch block

In [39]:
X_train.select_dtypes(include="number").head()

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,credit_score
0,31.0,44485.0,4,8000.0,8.83,0.18,668
1,24.0,35851.0,0,8725.0,12.18,0.24,645
2,28.0,121251.0,6,25000.0,11.01,0.21,569
3,23.0,76537.0,1,12000.0,14.96,0.16,538
4,24.0,53633.0,4,23975.0,11.01,0.45,678


## Strategy for each column : 

#### 1. Pass-Through Features
* **`credit_score`**: Pass through without changes.

#### 2. Feature Engineering & Scaling
* **`person_emp_exp`** (Employment Experience):
    * **New Feature (`is_exp`)**: Indicator variable where `person_emp_exp == 0`.
    * **New Feature (`log_positive_exp`)**: Applies `np.log1p(x)` if `person_emp_exp > 0`, else `0`.
    * **Scaling**: Apply Min-Max Scaling to `log_positive_exp` values greater than 0.

#### 3. Binning & Discretisation
* **`person_age`**: Bin into four age groups:
    * `0–25`
    * `25–30`
    * `35–40`
    * `40+`
* **`loan_int_rate`**: Bin into two interest rate groups:
    * `< 11`
    * `> 11`

#### 4. Mathematical Transformations
* **`person_income`**: Apply `LogTransformer`.
* **`loan_amnt`**: Apply `SqrtTransformer`.
* **`loan_percent_income`**: Apply `SqrtTransformer`.


## Custom Classes and Transformers for Numeric Columns

In [40]:
# Creating custom class for person_age
class CustomBinner (BaseEstimator,TransformerMixin):
    """
    A custom class that slices the specified column into intervals passes by user as a list.
    """
    
    def __init__(self,intervals):
        """
        intervals : Pass on the interval range as list ; for example : 
            range : 0-10,10-20,20-30,>30
            Pass this range as [10,20,30] (Do not pass the upper and lower bounds of the bins)
            The first  and last interval will automatically take all the remaining values below it and above it respectively 
        dtype : list[int,float]
        """
        self.intervals = intervals

        # Whenever modifying data we store it into a new variable , also this new variable must be given a defensive level
        self._clean_intervals = sorted(list(intervals))

    
    def fit(self,X,y=None):
        """
        Validates the dataslice passed and it's structure.
        """

        # Check whether the user has actually passed intervals to fit or not
        if not self.intervals:
            raise ValueError(f"CustomBinner needs at least one split point. Got: {self.intervals!r}")
        # Check all the variables passed in the list are of type : int,float
        if not all(isinstance(v, (int, float)) for v in self.intervals):
            raise TypeError(f"All values in 'intervals' must be numeric. Got: {self.intervals!r}")

        # First check whether the data structure passed ( can be : pandas,polars,etc ) contains the column attribute or not.
        " A simple validation to segregate these datatypes from np.array ."

        if hasattr(X,"columns"):
            self.feature_names_in_ = np.array(X.columns,dtype=object)
            " Extract the columns from the passed data structure."
        else:
            " The other case is it is a numpy array where cols dont have names so we assign temp feature names."
            # Check how many dimensions the data structure has
            num_cols = X.shape[1] if len(X.shape) > 1 else 1
            self.feature_names_in_ = np.array([f"X_{i}" for i in range(num_cols)], dtype=object)


        return self # important

    def transform(self,X):
        """ 
        Applying the custom transformation on the data structure passed.
        """
        if not hasattr(self, "feature_names_in_"):
            raise NotFittedError("This CustomBinner instance is not fitted yet.")
        
        X_arr = np.asarray(X, dtype=float)
        
        # np.digitize automatically handles underflow (<10) and overflow (>=30) seamlessly!
        return np.digitize(X_arr, bins=self._clean_intervals)        
    
    def get_feature_names_out(self, input_features=None):
        """ 
        A method to fetch the feature names : Original if passed or temp.
        """
        if input_features is None: # If the feature names were not  
            return self.feature_names_in_
        return np.array(input_features, dtype=object) # Basically to maintain pipeline naming sense
    

# Defining separate custom interval maps
age_splits = [25, 35, 40]
rate_splits = [11]

In [41]:
# Creating custom class for person_emp_exp

# Branch A: The Indicator Flag (Outputs 1.0 if experience == 0, else 0.0)
indicator_branch = FunctionTransformer(
    lambda X: (X == 0).astype(float),
    feature_names_out=lambda self, input_features: [f"{f}_is_exp" for f in input_features]
)

# Branch B: The Log Scaler (Applies np.log1p math, then fits a MinMaxScaler)
log_scale_branch = Pipeline([
    ('log_math', FunctionTransformer(
        np.log1p, 
        feature_names_out=lambda self, input_features: [f"{f}_log_positive_exp" for f in input_features]
    )),
    ('scaler', MinMaxScaler())
])

# Traffic Controller: Combines both branches to run in parallel on the same column
experience_engineer = FeatureUnion([
    ('indicator_branch', indicator_branch),
    ('log_scale_branch', log_scale_branch)
])

In [42]:
# Columns to undergo log transformation : 
log_cols = ['person_income']

# Columns to undergo sqrt transformation :
sqrt_cols = ['loan_amnt','loan_percent_income']

# Columns to undergo min-max transformation :
minmax_cols_input = ['person_income', 'loan_amnt', 'loan_percent_income',
                     'credit_score', 'person_age', 'person_education']

## Custom Transformers and Strategy for Object Columns

In [43]:
# We will make a column transformer for the encoders and create a custom Mapper function for our custom mapping cols

In [44]:
X_train.select_dtypes(include="object").head()

,person_gender,person_education,person_home_ownership,loan_intent,previous_loan_defaults_on_file
0,female,Associate,RENT,PERSONAL,Yes
1,female,Master,RENT,MEDICAL,Yes
2,female,High School,MORTGAGE,VENTURE,No
3,female,High School,RENT,MEDICAL,No
4,male,Associate,RENT,HOMEIMPROVEMENT,No


In [45]:
# To make sure the order for mapping cardinality values using OrdinalEncoder we will pass an explicit order to control behaviour
gender_order = ['female', 'male'] # Will map first value as 0 then next as 1 and so on...
file_order = ['No', 'Yes']

# Also the columns that we use OrdinalEncoder upon are : 
ord_cols = ["person_gender","previous_loan_defaults_on_file"]

# Now OneHotEncoded cols will be as decided : 
onc_cols = ["person_home_ownership","loan_intent"]

# Now handling our heirarchical col : person_education
heir_cols = ['person_education']

In [46]:
# Now creating a custom mapper for person_education
def home_ownership_mapper(X):

    hierarchy_map = {
                            'High School': 1,
                            'Associate': 2,
                            'Bachelor': 3,
                            'Master': 4,
                            'Doctorate' : 5
    }

    # 1. Convert to a NumPy array so the format is always consistent
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()
    
    # 2. Create a vectorized version of your dictionary lookup.
    # The .get(val, 0) handles missing values or typos by defaulting to 0.
    vector_lookup = np.vectorize(lambda val: hierarchy_map.get(val, 0))
    
    # 3. Apply it. This outputs the exact same 2D shape that came in.
    return vector_lookup(X)

# Wrap it up safely
home_ownership_transformer = FunctionTransformer(home_ownership_mapper, validate=False)

# Chainging Steps into a  PIPELINE 

In [47]:
# Column Transformer Sequential arranged 
col_transformer = ColumnTransformer(
        transformers = [
                            ('num_emp_features', experience_engineer, ['person_emp_exp']),

                            ('num_sqrt_transformer',
                                        Pipeline(
                                                  [
                                                        ('sqrt', FunctionTransformer(np.sqrt, validate=True)),
                                                        ('scaler', MinMaxScaler())
                                                   ]
                                                ),sqrt_cols),   # loan_amnt, loan_percent_income

                            ('num_log_transformer',
                                        Pipeline(
                                                  [
                                                        ('log', FunctionTransformer(np.log1p)),
                                                        ('scaler', MinMaxScaler())
                                                   ]
                                                ),log_cols),    # person_income

                            ('num_age_binner', CustomBinner(intervals=age_splits), ['person_age']),
                            ('num_rate_binner', CustomBinner(intervals=rate_splits), ['loan_int_rate']),
                            ("obj_enc_ord",OrdinalEncoder(
                                                                categories=[gender_order,file_order],
                                                                handle_unknown='use_encoded_value',
                                                                unknown_value=-1
                            ),ord_cols),
                            ("obj_enc_onc",OneHotEncoder(drop='first',handle_unknown='error'),onc_cols),
                            ("obj_enc_heir",home_ownership_transformer,heir_cols),
                            ('num_credit_score', MinMaxScaler(), ['credit_score']),
        ], 
        remainder='drop'
)

In [48]:
# Create the final pipeline
lor_pipeline = Pipeline(steps=[
    ('preprocessor', col_transformer),
    ('lor', LogisticRegression(max_iter=2000))
])

# Fit everything 
lor_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('lor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_emp_features', ...), ('num_sqrt_transformer', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output